# 07 · Conclusions, HTML Report & Dashboard
**Phishing E-mail Detection: A Leakage-Controlled Benchmark of Classical, Deep and Transformer Models**  
*Parts VIII-IX · Sections 26-28 & Appendix* · MSc Data Science final project · Dr. Spoorthi H S

**In this notebook**
- Auto-generated answer to the research question, read live from the run's results
- Limitations and future work
- Standalone HTML report and an ipywidgets phishing-risk dashboard
- Change log (Appendix A)

> **How to use these notebooks.** The outputs below come from one complete run of the pipeline on a Kaggle Tesla T4 GPU. Every part builds on objects created earlier (the leak-free split, fitted models and their predictions), so the part notebooks are for reading and review. To reproduce the results, run [`00_full_pipeline_run_all.ipynb`](00_full_pipeline_run_all.ipynb) top to bottom.

[← RAG, Latency, Deployment & LangGraph SOC Agent](06_rag_latency_deployment_soc_agent.ipynb)

---

<a id="partVIII"></a>

<div style="background:#1D3557;color:#fff;padding:16px 24px;border-radius:12px;margin:34px 0 10px 0">
<div style="font-size:0.78em;letter-spacing:.16em;text-transform:uppercase;opacity:.85">Part VIII</div>
<div style="font-size:1.7em;font-weight:800;line-height:1.2">Conclusions</div>
<div style="font-size:0.95em;opacity:.92;margin-top:4px">Section 26 &nbsp;&middot;&nbsp; Research questions: RQ1-RQ8</div>
</div>


<a id="sec26"></a>

## 26. Conclusions & Answer to the Research Question

The code cell below generates the headline answer directly from this run's own results (`RESULTS`, `ci_df`, `BEST_MODEL_NAME`), so the quoted numbers cannot drift from the outputs above it.

**Key methodological points carried through this pipeline:**
- The leakage audit (Section 4) and the grouped re-split apply to **every** model, not just the Transformers as in the original notebooks, so the comparison in Section 11 is like-for-like.
- `source` was excluded as a model input (Section 4.4) because several sources are near-perfectly single-class (Section 3.2). Section 20 gives a quantified check of that risk and Section 20.2 tests how much performance depends on source-identifying tokens.
- **Model selection uses validation data only:** the top-2 baselines and Transformers (8.5, 9.4), the best model (14), the calibration method (15.2) and the decision threshold (15.3). The test set is used to *report* results. An earlier version of this notebook ranked the final model and the calibration method by test-set scores; that was corrected (see Appendix A).
- Saved Transformer checkpoints can be reloaded (Section 12.1), so a Transformer can be the selected model and take part in the calibration, explainability and robustness analyses. The out-of-fold stack (12.2) covers only the two classical models.

**What the extended analysis (Sections 15-24) added beyond the original benchmark:**
- **Calibration & thresholding (15):** whether raw probabilities are trustworthy, and cost-sensitive operating points rather than a bare 0.5 cut.
- **Explainability (16):** SHAP for the linear baseline; Integrated Gradients for the hybrid model and the selected Transformer need the optional `captum` package.
- **Tactic detection (17) & threat-intel extraction (18):** engineering extensions that turn the label into analyst-facing output; their evaluation is limited (see 26.1).
- **Ablations (19):** attention pooling, feature fusion, text cleaning and individual engineered features, each quantified rather than assumed to matter.
- **Cross-source generalisation (20), source-artefact test (20.2) and adversarial robustness (21, 21.3):** independent, quantified answers to "how far can we trust this beyond the exact test distribution?", including a normalisation defence.
- **RAG explanation (22), efficiency (23), deployment artifacts (24) and the SOC agent (25):** engineering extensions built on the benchmark and evaluated separately and less exhaustively.


In [91]:
# Auto-generated research-question answer -- every number below is pulled directly
# from this run's own RESULTS / ci_df / BEST_MODEL_NAME / bench_df, never hand-typed,
# so this cell can be re-run after any configuration change (QUICK_RUN, RUN_TRANSFORMERS,
# RUN_HP_SEARCH, ...) and the narrative always matches the actual numbers above it.

def _fmt_list(names):
    return ", ".join(display_name(n) for n in names) if names else "none available this run"

def _fmt(v, digits=4):
    return f"{v:.{digits}f}" if isinstance(v, (int, float)) and not isinstance(v, bool) and v == v else "n/a"

_best_test = RESULTS.get(f"{BEST_MODEL_NAME}_test", {})
_best_test_f1 = _best_test.get("f1")
_best_test_auc = _best_test.get("roc_auc")

_ci_line = "bootstrap CI not available (fewer than 2 candidates were compared in Section 13)"
if "ci_df" in globals() and BEST_MODEL_NAME in ci_df.index:
    _row = ci_df.loc[BEST_MODEL_NAME]
    _ci_line = f"95% bootstrap CI [{_fmt(_row['f1_lo'])}, {_fmt(_row['f1_hi'])}] (Section 13)"

_sig_line = "not computed (fewer than 2 finalists were available to compare)"
if "sig_test" in globals() and "top2" in globals() and len(top2) >= 2:
    _verdict = "WAS" if sig_test["p_value"] < 0.05 else "was NOT"
    _sig_line = (f"{display_name(top2[0])} vs {display_name(top2[1])}: observed F1 difference "
                 f"{sig_test['mean_diff']:+.4f}, paired bootstrap p={_fmt(sig_test['p_value'])} "
                 f"-> this margin {_verdict} statistically significant at alpha=0.05")

_ensemble_line = "not computed (fewer than 2 finalists were available to stack)"
if "ENSEMBLE_BASE_MODELS" in globals() and ENSEMBLE_BASE_MODELS:
    _stack_f1 = RESULTS.get("EnsembleStack_test", {}).get("f1")
    _indiv_f1s = [RESULTS.get(f"{n}_test", {}).get("f1", 0) for n in ENSEMBLE_BASE_MODELS]
    if _stack_f1 is not None and _indiv_f1s:
        _best_indiv_f1 = max(_indiv_f1s)
        _did = "DID improve on" if _stack_f1 > _best_indiv_f1 else "did NOT improve on"
        _ensemble_line = (f"the stacked ensemble of {_fmt_list(ENSEMBLE_BASE_MODELS)} scored test "
                           f"F1={_fmt(_stack_f1)}, which {_did} its own best base model "
                           f"(F1={_fmt(_best_indiv_f1)})")

_thresh_line = "not computed (Section 15.3 threshold search did not run)"
if "OPTIMAL_THRESHOLD" in globals() and "y_true_cal" in globals():
    from sklearn.metrics import precision_score as _ps, recall_score as _rs
    _prec_d = _ps(y_true_cal, y_pred_default, zero_division=0)
    _rec_d = _rs(y_true_cal, y_pred_default, zero_division=0)
    _prec_o = _ps(y_true_cal, y_pred_optimal, zero_division=0)
    _rec_o = _rs(y_true_cal, y_pred_optimal, zero_division=0)
    _thresh_line = (f"moving from the default 0.5 threshold to the SOC-cost-optimised "
                     f"{_fmt(OPTIMAL_THRESHOLD, 3)} (Section 15.3) changed recall {_fmt(_rec_d, 3)} -> "
                     f"{_fmt(_rec_o, 3)} and precision {_fmt(_prec_d, 3)} -> {_fmt(_prec_o, 3)} on the "
                     "held-out test set")

_efficiency_line = "not computed (Section 23 efficiency benchmark did not run)"
if "bench_df" in globals() and len(bench_df):
    _match = bench_df[bench_df["model"] == BEST_MODEL_NAME]
    if len(_match):
        _r = _match.iloc[0]
        _size = _fmt(_r['size_mb'], 2)
        _efficiency_line = (f"{display_name(BEST_MODEL_NAME)} costs {_fmt(_r['batched_ms_per_item'], 3)} "
                             f"ms/email (batched inference)"
                             + (f" and {_size} MB on disk" if _size != "n/a"
                                else " (on-disk size was not measured for Transformer checkpoints)")
                             + " (Section 23)")

print("=" * 100)
print("RESEARCH QUESTION -- ANSWER (every figure below is read live from this run's own results)")
print("=" * 100)
print(f"""
Across the finalist pool evaluated on the identical leak-free split (Section 4) -- the
top-2 classical baselines ({_fmt_list(TOP2_BASELINES)}), the top-2 fine-tuned Transformers
({_fmt_list(TOP2_TRANSFORMERS)}), and the custom Attention-BiGRU + engineered-features model
-- {display_name(BEST_MODEL_NAME)} achieved the highest held-out test F1 ({_fmt(_best_test_f1)},
{_ci_line}) and ROC-AUC ({_fmt(_best_test_auc)}).

Selection protocol: {display_name(BEST_MODEL_NAME)} was selected on validation F1 (Section 14), its calibration method and decision threshold were chosen on validation data (Section 15), and test metrics are reported only after those choices were fixed.

Statistical significance of the top-2 gap (Section 13): {_sig_line}.

Ensembling (Section 12): {_ensemble_line}.

Security-aware thresholding (Section 15): {_thresh_line}.

Deployment cost of the winning model (Section 23): {_efficiency_line}.
""")

# Data-provenance note is ALWAYS printed (not just under QUICK_RUN) and is read live from
# DATA_SOURCE_NOTE/USED_SYNTHETIC_DATA (Section 2) -- never hand-typed -- so this claim can never
# silently go stale if a real dataset is attached in a later run and this cell isn't otherwise edited.
print(f"[DATA PROVENANCE] {DATA_SOURCE_NOTE}")
if USED_SYNTHETIC_DATA:
    print("[SCOPE NOTE] No real dataset was found under /kaggle/input, /kaggle/working, or '.' for "
          "this run, so Section 2 generated a small SYNTHETIC placeholder sample instead. Every "
          "number above is a code-correctness smoke test, NOT a research finding -- attach the real "
          "dataset (e.g. Kaggle's 'Phishing Email Dataset' by naserabdullahalam, or your own "
          "train/val/test CSVs) and re-run for numbers worth reporting.")

if not RUN_TRANSFORMERS:
    print("[SCOPE NOTE] RUN_TRANSFORMERS was False for this run (Section 1) -- this sandbox has no "
          "route to huggingface.co to download pretrained weights, so no Transformer finalist "
          "competed here. Re-run with RUN_TRANSFORMERS=True on a machine with Hugging Face Hub "
          "access (e.g. Kaggle/Colab with a GPU) to get a genuine classical-vs-Transformer answer; "
          "every Transformer code path above is intact and ready to run there unmodified.")
if QUICK_RUN:
    print("[SCOPE NOTE] QUICK_RUN was True for this run -- small grids and few epochs throughout, "
          "regardless of which data source was used (see the data-provenance note above). The "
          "numbers above demonstrate that every code path runs correctly end-to-end, not a real "
          "research finding; set QUICK_RUN=False with the real dataset attached for numbers worth "
          "reporting.")


RESEARCH QUESTION -- ANSWER (every figure below is read live from this run's own results)

Across the finalist pool evaluated on the identical leak-free split (Section 4) -- the
top-2 classical baselines (TF-IDF + Linear SVM, TF-IDF + Logistic Regression), the top-2 fine-tuned Transformers
(RoBERTa-base, DistilBERT), and the custom Attention-BiGRU + engineered-features model
-- RoBERTa-base achieved the highest held-out test F1 (0.9943,
95% bootstrap CI [0.9929, 0.9956] (Section 13)) and ROC-AUC (0.9997).

Selection protocol: RoBERTa-base was selected on validation F1 (Section 14), its calibration method and decision threshold were chosen on validation data (Section 15), and test metrics are reported only after those choices were fixed.

Statistical significance of the top-2 gap (Section 13): RoBERTa-base vs DistilBERT: observed F1 difference -0.0005, paired bootstrap p=0.5010 -> this margin was NOT statistically significant at alpha=0.05.

Ensembling (Section 12): the stacked ensemble

### 26.1 Limitations and Future Work

**Limitations**
1. **Dataset validity.** The pool combines older spam/legitimate-mail corpora (CEAS-08, Enron, Ling, Nazario, Nigerian Fraud and an unlabelled-source residual). Two sources are 100% phishing, "phishing" includes advance-fee fraud and bulk spam, and the most discriminative tokens include corpus identifiers (`enron`, `dynegy`, `spambayes`). In-distribution F1 of about 0.995 is therefore optimistic: leave-one-source-out F1 was 0.82-0.88 for TF-IDF + Logistic Regression (Section 20), and Section 20.2 tests the dependence on source-identifying tokens.
2. **Representativeness.** The data has no email headers, attachments or URL reputation, and does not cover modern brand-impersonation, QR-code or LLM-written phishing. Results should not be read as performance on current real-world mail.
3. **Statistical scope.** Each model was trained once (one seed, one split, a two-value learning-rate grid for the Transformers). Bootstrap intervals capture test-set sampling noise, not training variance, so gaps of a few thousandths of F1 (e.g. DistilBERT vs RoBERTa, p = 0.50) should not be over-interpreted.
4. **Adversarial robustness.** Obfuscation caused F1 drops of 0.14-0.31 in the captured run. Section 21.3 adds a non-adaptive normalisation defence; adaptive attackers, and other evasion routes (image-only mail, hidden text), were not tested.
5. **Operating point.** The 25:1 false-negative/false-positive cost ratio is an assumption, and the cost-optimal threshold (0.010) hit the edge of the search grid (see 15.3 and 15.3b). The reference deployment (Section 24) serves TF-IDF + Logistic Regression, not the selected model.
6. **Extensions.** The tactic classifier (17) is trained on labels derived from keyword lexicons, so its F1 measures how well the lexicon is reproduced, not agreement with human annotation. The IOC extractor (18) uses heuristics, and the RAG/agent sections (22, 25) are evaluated on tiny sets (see the evaluation-status note after Section 25.6).
7. **Ethics and privacy.** The corpora contain real people's email text. Any deployment would need data-protection review, and a false negative on a targeted attack has a very different cost from the average assumed here.

**Future work**
- Re-run the pipeline on a modern, header-aware phishing dataset and evaluate on a temporally later hold-out.
- Repeat training over several seeds/splits and report mean and spread; add source-grouped cross-validation.
- Explore additional pretrained encoders and a wider learning-rate search; consider LoRA variants alongside full fine-tuning.
- Evaluate the tactic labels and the agent against human annotation, with a real LLM and groundedness measured on a larger email set.
- Evaluate adaptive adversarial attacks and adversarial training in addition to input normalisation.

<a id="partIX"></a>

<div style="background:#5C677D;color:#fff;padding:16px 24px;border-radius:12px;margin:34px 0 10px 0">
<div style="font-size:0.78em;letter-spacing:.16em;text-transform:uppercase;opacity:.85">Part IX</div>
<div style="font-size:1.7em;font-weight:800;line-height:1.2">Supplementary Deliverables</div>
<div style="font-size:0.95em;opacity:.92;margin-top:4px">Sections 27-28 and Appendix A &nbsp;&middot;&nbsp; Research questions: -</div>
</div>


<a id="sec27"></a>

## 27. Standalone HTML Report

Sections 1-26 are the full research + deployment notebook. The cell below
writes a single self-contained `phishing_detection_report.html` file -- a
designed, presentation-ready dashboard covering the model leaderboard,
statistical significance, a worked sample detection report (prediction +
SHAP attribution + retrieved evidence + analyst guidance), the adversarial
and cross-source robustness results, tactic/IOC extraction, and the
FastAPI/Docker/Streamlit deployment guide -- so results can be shared with
a reviewer or a SOC stakeholder without asking them to run the notebook.


In [92]:
"""Standalone HTML report -- a self-contained, presentation-ready summary of
this notebook's evaluation results and deployment guide, meant to be opened
directly in a browser (no dependencies, no server) or shared with a reviewer
who won't run the notebook. Every figure and every model name in it is read
live from this run's own variables (BEST_MODEL_NAME, RESULTS, ci_df, bench_df,
cross_source_df, robustness_df, tactic_report, ioc_df, ...) -- nothing is
hand-typed, so whichever model wins the Section 14 validation-F1 selection is
the model this report describes and labels as the winner."""
import json as _json
from pathlib import Path as _Path

REPORT_DIR = WORKING_DIR / "report"
REPORT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------------
# Small formatting helpers
# ------------------------------------------------------------------
def _f(v, d=4):
    try:
        v = float(v)
        return f"{v:.{d}f}" if v == v else "n/a"  # v == v is False for NaN
    except (TypeError, ValueError):
        return "n/a"


def _esc(s):
    return _dash_html.escape(str(s)) if "_dash_html" in globals() else str(s).replace("<", "&lt;").replace(">", "&gt;")


import html as _dash_html

CHAMPION = BEST_MODEL_NAME
CHAMPION_DISP = display_name(CHAMPION)
_champ_test = RESULTS.get(f"{CHAMPION}_test", {})
_champ_val = RESULTS.get(f"{CHAMPION}_val", {})


def _bench_row(name):
    if "bench_df" in globals() and len(bench_df):
        m = bench_df[bench_df["model"] == name]
        if len(m):
            return m.iloc[0]
    return None


# ------------------------------------------------------------------
# 1. Leaderboard (Section 11/13 finalists, ranked by test F1; the
#    "winner" tag marks CHAMPION -- the model actually selected on
#    VALIDATION F1 in Section 14, which need not be the top test-F1 row).
# ------------------------------------------------------------------
_lb_models = [m for m in FINALISTS if f"{m}_test" in RESULTS]
_lb_models = sorted(_lb_models, key=lambda n: -RESULTS[f"{n}_test"].get("f1", 0))

_lb_rows = []
for name in _lb_models:
    m = RESULTS[f"{name}_test"]
    b = _bench_row(name)
    train_s = f"{m['train_seconds']:.1f}s" if m.get("train_seconds") is not None else "\u2014"
    params = f"{b['n_params'] / 1e6:.1f}M" if (b is not None and b.get("n_params")) else "\u2014"
    winner = (name == CHAMPION)
    _lb_rows.append(f"""
        <tr{' class="winner"' if winner else ''}>
          <td class="hl">{_esc(display_name(name))}{' <span class="tag win">winner</span>' if winner else ''}</td>
          <td class="num">{_f(m.get('accuracy'))}</td><td class="num">{_f(m.get('precision'))}</td>
          <td class="num">{_f(m.get('recall'))}</td><td class="num-hl">{_f(m.get('f1'))}</td>
          <td class="num">{_f(m.get('roc_auc'))}</td><td class="num">{train_s}</td><td class="num">{params}</td>
        </tr>""")
LEADERBOARD_ROWS_HTML = "".join(_lb_rows)

# Significance callout: compare the actual top-2 (by validation F1, Section 13)
_sig_html = f"{CHAMPION_DISP} was selected as the deployed model on validation F1 (Section 14)."
if "top2" in globals() and len(top2) >= 2 and "sig_test" in globals():
    _n1, _n2 = display_name(top2[0]), display_name(top2[1])
    _diff = sig_test.get("mean_diff", float("nan"))
    _p = sig_test.get("p_value", float("nan"))
    _sig_word = "not significant" if (isinstance(_p, (int, float)) and _p == _p and _p >= 0.05) else "significant"
    _sig_html = (
        f'<b>Is the winner actually better, or is that noise?</b> A paired bootstrap significance '
        f'test on the top-2 validation gap ({_esc(_n1)} vs. {_esc(_n2)}, {_diff:+.4f} F1) returns '
        f'<code>p = {_f(_p, 3)}</code> \u2014 {_sig_word} at &alpha; = 0.05. {_esc(CHAMPION_DISP)} is the '
        f'model actually shipped as <code>BEST_MODEL_NAME</code> because it had the highest '
        f'<b>validation</b> F1 (Section 14) \u2014 the held-out test set (this table) is reported for '
        f'transparency only and was not used to choose it.'
    )

# ------------------------------------------------------------------
# 2. Ensembling reference cards
# ------------------------------------------------------------------
def _ensemble_card(title, tag, base_models_var, stack_key):
    if base_models_var not in globals() or not globals()[base_models_var]:
        return f'<div class="card"><h3>{title} <span class="tag">{tag}</span></h3><p>Not computed this run.</p></div>'
    bases = globals()[base_models_var]
    stack_f1 = RESULTS.get(f"{stack_key}_test", {}).get("f1")
    indiv = [RESULTS.get(f"{n}_test", {}).get("f1", 0) for n in bases]
    if stack_f1 is None or not indiv:
        return f'<div class="card"><h3>{title} <span class="tag">{tag}</span></h3><p>Not computed this run.</p></div>'
    best_indiv = max(indiv)
    cls = "good" if stack_f1 > best_indiv else "warn"
    verdict = "edges past" if stack_f1 > best_indiv else "does not beat"
    return f"""
    <div class="card">
      <h3>{title} <span class="tag">{_esc(tag)}</span></h3>
      <div class="metric {cls}">F1 {_f(stack_f1)}</div>
      <p>Stacking {_esc(', '.join(display_name(n) for n in bases))}: this {verdict} the best individual base model (F1 {_f(best_indiv)}).</p>
    </div>"""

ENSEMBLE_CARDS_HTML = (
    _ensemble_card("Primary OOF stack", "classical pair", "ENSEMBLE_BASE_MODELS", "EnsembleStack")
    + _ensemble_card("Extended reference stack", "all finalists", "ENSEMBLE_BASE_MODELS_EXT", "EnsembleStackExt")
)

# ------------------------------------------------------------------
# 3. Inference cost table (Section 23)
# ------------------------------------------------------------------
_cost_rows = []
if "bench_df" in globals() and len(bench_df):
    for _, r in bench_df.sort_values("batched_ms_per_item").iterrows():
        winner = (r["model"] == CHAMPION)
        size = f"{r['size_mb']:.2f} MB" if r.get("size_mb") == r.get("size_mb") and r.get("size_mb") is not None else "\u2014"
        _cost_rows.append(f"""
        <tr{' class="winner"' if winner else ''}><td class="hl">{_esc(display_name(r['model']))}</td>
        <td class="num">{_f(r['single_item_ms_p95'], 1)} ms</td>
        <td class="num">{_f(r['batched_ms_per_item'], 2)} ms</td>
        <td class="num">{_f(r['throughput_emails_per_sec'], 0)} / sec</td>
        <td class="num">{size}</td></tr>""")
COST_ROWS_HTML = "".join(_cost_rows) if _cost_rows else '<tr><td colspan="5">Not computed this run (Section 23).</td></tr>'

_champion_bench = _bench_row(CHAMPION)
_tfidf_bench = _bench_row("TFIDF_LogReg")
_deploy_cost_note = (
    f"The reference deployment below deliberately ships <b>TF-IDF + Logistic Regression</b>, not the "
    f"validation-selected model (<b>{_esc(CHAMPION_DISP)}</b>) \u2014 see Section 24's design note for why "
    f"a small, dependency-light service is preferred for a low-latency mail gateway."
)
if _champion_bench is not None and _tfidf_bench is not None:
    try:
        _ratio_lat = _champion_bench["batched_ms_per_item"] / max(_tfidf_bench["batched_ms_per_item"], 1e-9)
        _f1_cost = (_champ_test.get("f1", float("nan")) - RESULTS.get("TFIDF_LogReg_test", {}).get("f1", float("nan")))
        _deploy_cost_note = (
            f"This is exactly why the deployment reference below ships <b>TF-IDF + Logistic Regression</b>, "
            f"not the validation-selected model (<b>{_esc(CHAMPION_DISP)}</b>) \u2014 it runs roughly "
            f"<b>{_ratio_lat:.1f}\u00d7</b> faster per email in batch, for a {_f(abs(_f1_cost), 3)}-point F1 "
            f"difference, which matters at real gateway volume (Section 23/24)."
        )
    except Exception:
        pass

# ------------------------------------------------------------------
# 4. Sample detection report -- uses the CHAMPION model's own score
#    (y_proba_best / y_pred_best from Section 14.1), real tactic and
#    IOC extraction for that row, and the same retrieval used by
#    Section 22's generate_explanation_report().
# ------------------------------------------------------------------
DEMO_HTML = '<p>Not available this run (needs Section 14.1\'s y_proba_best and Section 18\'s IOC extraction to have run).</p>'
try:
    _demo_candidates = np.where(y_pred_best == 1)[0]
    _demo_idx = int(_demo_candidates[0]) if len(_demo_candidates) else 0
    _demo_text = str(test_df["text"].iloc[_demo_idx])
    _demo_proba = float(y_proba_best[_demo_idx])
    _demo_thresh = OPTIMAL_THRESHOLD if "OPTIMAL_THRESHOLD" in globals() else 0.5
    _demo_decision = "PHISHING" if _demo_proba >= _demo_thresh else "LEGIT"

    _demo_tactics = []
    if "test_tactic_pred" in globals() and "TACTIC_NAMES" in globals():
        _demo_tactics = [t for t, v in zip(TACTIC_NAMES, test_tactic_pred[_demo_idx]) if v]

    _demo_iocs = extract_iocs(_demo_text) if "extract_iocs" in globals() else {"urls": [], "emails": [], "ips": []}
    _demo_n_iocs = sum(len(v) for v in _demo_iocs.values())

    _demo_evidence_html = ""
    if "retrieve_evidence" in globals():
        _query = _demo_text + " " + " ".join(_demo_tactics)
        for kb_id, kb_text, sim in retrieve_evidence(_query, k=3):
            _demo_evidence_html += f"""
      <div class="evidence-item">
        <div class="ev-head"><b>{_esc(kb_id)}</b><span>similarity {sim:.2f}</span></div>
        <p>{_esc(kb_text)}</p>
      </div>"""

    _demo_tokens_html = ""
    if "global_shap_df" in globals() and CHAMPION == "TFIDF_LogReg":
        for rank, (_, row) in enumerate(global_shap_df.head(5).iterrows(), start=1):
            _demo_tokens_html += (
                f'<tr><td class="num">{rank}</td><td class="hl">{_esc(row["token"])}</td>'
                f'<td class="num" style="color:var(--sage)">{row["mean_abs_shap"]:+.4f}</td></tr>'
            )
        _tokens_block = f"""
      <div class="report-sub-head">Global token importance -- {_esc(CHAMPION_DISP)} (Section 16.1)</div>
      <div class="table-wrap">
        <table><thead><tr><th>Rank</th><th>Token</th><th>Mean |SHAP|</th></tr></thead>
        <tbody>{_demo_tokens_html}</tbody></table>
      </div>"""
    else:
        _tokens_block = (
            f'<div class="callout">Token-level attribution for <b>{_esc(CHAMPION_DISP)}</b> was not computed '
            f'in this run (Section 16 covers SHAP for TF-IDF + Logistic Regression and Integrated Gradients '
            f'for the winning Transformer, when <code>RUN_SHAP</code>/<code>RUN_IG</code> are enabled).</div>'
        )

    _risk_class = "risk-chip" if _demo_decision == "PHISHING" else 'risk-chip" style="background:rgba(127,217,154,0.1); border-color:rgba(127,217,154,0.35); color:var(--risk-low)'
    DEMO_HTML = f"""
  <div class="report-card">
    <div class="report-top">
      <div class="report-input">
        <span class="lbl">Test row #{_demo_idx} \u2014 held-out email excerpt</span>
        "{_esc(_demo_text[:180])}"
      </div>
      <div class="{_risk_class}"><span class="dot"></span> {_demo_decision} \u2014 score {_demo_proba:.3f}</div>
    </div>
    <div class="report-body">
      <div class="table-wrap" style="margin-top:14px;">
        <table>
          <thead><tr><th>Model</th><th>Decision</th><th>Score</th><th>Threshold</th><th>IOCs found</th><th>Tactics</th></tr></thead>
          <tbody>
            <tr><td class="hl">{_esc(CHAMPION_DISP)}</td>
                <td><span class="tag {'win' if _demo_decision == 'PHISHING' else ''}">{_demo_decision}</span></td>
                <td class="num">{_demo_proba:.3f}</td><td class="num">{_f(_demo_thresh, 3)}</td>
                <td class="num">{_demo_n_iocs}</td><td class="num">{', '.join(_demo_tactics) or 'none flagged'}</td></tr>
          </tbody>
        </table>
      </div>
      {_tokens_block}
      <div class="report-sub-head">Retrieved supporting evidence (curated SOC knowledge base)</div>
      {_demo_evidence_html or '<p>No evidence retrieved.</p>'}
    </div>
  </div>"""
except Exception as _e:
    DEMO_HTML = f'<p>[INFO] Sample detection demo unavailable this run: {_esc(_e)}</p>'

# ------------------------------------------------------------------
# 5. Threshold & error analysis (Section 15.3 / 14.2)
# ------------------------------------------------------------------
_thresh_cards = ""
try:
    from sklearn.metrics import precision_score as _ps2, recall_score as _rs2
    _rec_d = _rs2(y_true_cal, y_pred_default, zero_division=0)
    _rec_o = _rs2(y_true_cal, y_pred_optimal, zero_division=0)
    _prec_d = _ps2(y_true_cal, y_pred_default, zero_division=0)
    _prec_o = _ps2(y_true_cal, y_pred_optimal, zero_division=0)
    _thresh_cards = f"""
    <div class="card">
      <h3>Cost-optimal threshold</h3>
      <div class="metric">{_f(OPTIMAL_THRESHOLD, 3)}</div>
      <p>Chosen to minimise expected validation-set cost under the SOC cost model (Section 15.3).</p>
    </div>
    <div class="card">
      <h3>Recall shift</h3>
      <div class="metric good">{_f(_rec_d, 3)} \u2192 {_f(_rec_o, 3)}</div>
      <p>Phishing recall at default vs. tuned threshold, {_esc(CHAMPION_DISP)}, test set.</p>
    </div>
    <div class="card">
      <h3>Precision trade-off</h3>
      <div class="metric warn">{_f(_prec_d, 3)} \u2192 {_f(_prec_o, 3)}</div>
      <p>The accepted cost, in false alarms, of catching more real phishing.</p>
    </div>"""
except Exception:
    _thresh_cards = '<div class="card"><p>Not computed this run (Section 15.3).</p></div>'

_error_cards = ""
try:
    _n_fn, _n_pos = len(false_negatives), (err_df["y_true"] == 1).sum()
    _n_fp, _n_neg = len(false_positives), (err_df["y_true"] == 0).sum()
    _error_cards = f"""
    <div class="card">
      <h3>False negatives \u2014 missed phishing</h3>
      <div class="metric bad">{_n_fn} / {_n_pos}</div>
      <p>Phishing emails {_esc(CHAMPION_DISP)} scored below the operating threshold.</p>
    </div>
    <div class="card">
      <h3>False positives \u2014 legit flagged</h3>
      <div class="metric warn">{_n_fp} / {_n_neg}</div>
      <p>Legitimate emails {_esc(CHAMPION_DISP)} scored above the operating threshold.</p>
    </div>"""
except Exception:
    _error_cards = '<div class="card"><p>Not computed this run (Section 14.2).</p></div>'

# ------------------------------------------------------------------
# 6. Robustness: cross-source generalisation + adversarial obfuscation
#    (Section 20 / 21 -- robustness_df is already computed against
#    CHAMPION == BEST_MODEL_NAME, not hand-picked)
# ------------------------------------------------------------------
_cross_source_rows_html = ""
if "cross_source_df" in globals() and len(cross_source_df):
    for _, r in cross_source_df.iterrows():
        _cross_source_rows_html += (
            f'<tr><td class="hl">{_esc(r["held_out_source"])}</td><td class="num">{int(r["n_held_out"]):,}</td>'
            f'<td class="num">{_f(r["f1"], 3)}</td><td class="num">{_f(r["recall"], 3)}</td>'
            f'<td class="num">{_f(r["precision"], 3)}</td><td class="num">{_f(r["roc_auc"], 3)}</td></tr>'
        )
else:
    _cross_source_rows_html = '<tr><td colspan="6">Not computed this run (Section 20).</td></tr>'

_obf_bars_html = ""
_obf_summary = "Not computed this run (Section 21)."
if "robustness_df" in globals() and len(robustness_df):
    _max_f1 = robustness_df["f1"].max() or 1.0
    for _, r in robustness_df.iterrows():
        pct = max(0.0, min(100.0, 100.0 * r["f1"] / _max_f1))
        _obf_bars_html += (
            f'<div class="bar-row"><div class="bar-label">{_esc(r["perturbation"].capitalize())}</div>'
            f'<div class="bar-track"><div class="bar-fill" style="width:{pct:.1f}%"></div></div>'
            f'<div class="bar-val">{_f(r["f1"], 3)}</div></div>'
        )
    _worst = robustness_df.loc[robustness_df["f1_drop"].idxmax()]
    _obf_summary = (
        f"The worst single technique against {_esc(CHAMPION_DISP)} was '{_esc(_worst['perturbation'])}' "
        f"(F1 drop of {_f(_worst['f1_drop'], 3)}). Mitigation: fold homoglyphs and leetspeak back to plain "
        f"ASCII in a normalisation pass before scoring (Section 21.3)."
    )

# ------------------------------------------------------------------
# 7. Signals: tactic detection + IOC extraction (Sections 17/18 --
#    already model-agnostic classifiers, but the IOC table is built
#    from CHAMPION's own flagged rows, Section 18.2)
# ------------------------------------------------------------------
_tactic_rows_html = ""
if "tactic_report" in globals() and len(tactic_report):
    for _, r in tactic_report.sort_values("test_f1_tuned", ascending=False).iterrows():
        _tactic_rows_html += (
            f'<tr><td class="hl">{_esc(str(r["tactic"]).replace("_", " ").title())}</td>'
            f'<td class="num">{_f(r["test_f1_tuned"], 3)}</td>'
            f'<td class="num">{r["test_prevalence"]:.1%}</td></tr>'
        )
else:
    _tactic_rows_html = '<tr><td colspan="3">Not computed this run (Section 17).</td></tr>'

_ioc_block = '<p>Not computed this run (Section 18.2).</p>'
if "ioc_df" in globals() and len(ioc_df):
    _n_flagged = int(len(flagged_idx)) if "flagged_idx" in globals() else int(ioc_df["row"].nunique())
    _type_counts = ioc_df.groupby("type")["value"].nunique()
    _ioc_block = f"""
      <div class="metric" style="margin-top:2px;">{_n_flagged:,} emails</div>
      <p style="margin-bottom:14px;">flagged phishing by {_esc(CHAMPION_DISP)}, yielding <b>{ioc_df.drop_duplicates(['type','value']).shape[0]:,} unique indicators</b> ready for a blocklist/threat-intel feed:</p>
      {"".join(f'<div class="bar-row"><div class="bar-label">{_esc(t.capitalize())}s</div><div class="bar-track"><div class="bar-fill alt" style="width:{100*n/_type_counts.max():.0f}%"></div></div><div class="bar-val">{n}</div></div>' for t, n in _type_counts.sort_values(ascending=False).items())}
    """

# ------------------------------------------------------------------
# 8. Conclusion paragraph
# ------------------------------------------------------------------
_ci_line = ""
if "ci_df" in globals() and CHAMPION in ci_df.index:
    _row = ci_df.loc[CHAMPION]
    _ci_line = f", 95% bootstrap CI [{_f(_row.get('f1_lo'))}, {_f(_row.get('f1_hi'))}]"

CONCLUSION_HTML = (
    f"Across the finalist pool evaluated on the identical leak-free split, <b>{_esc(CHAMPION_DISP)} was "
    f"selected as the deployed model (validation F1 {_f(_champ_val.get('f1'))}, Section 14)</b>, reaching "
    f"held-out test F1 {_f(_champ_test.get('f1'))}{_ci_line} and ROC-AUC {_f(_champ_test.get('roc_auc'))}. "
    f"{_sig_html} The model's F1 drops under source shift and under adversarial text obfuscation "
    f"(Sections 20-21), both measured directly rather than assumed away -- exactly the kind of finding "
    f"that separates a leaderboard score from a production-ready system."
)

HERO_STAT_F1 = _f(_champ_test.get("f1"))
HERO_STAT_MODEL = _esc(CHAMPION_DISP)
HERO_STAT_THRESH = _f(OPTIMAL_THRESHOLD, 3) if "OPTIMAL_THRESHOLD" in globals() else "n/a"
HERO_STAT_N = f"{len(full_df):,}" if "full_df" in globals() else "n/a"
HERO_STAT_LATENCY = (f"{_champion_bench['batched_ms_per_item']:.2f}ms" if _champion_bench is not None else "n/a")

print("Report data assembled for champion model:", CHAMPION_DISP)
_HTML_HEAD = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Phishing Detection System — Evaluation &amp; Deployment Report</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500;600;700&display=swap" rel="stylesheet">
<style>
:root{
  --bg:#0A0E13; --bg-panel:#111823; --bg-panel-raised:#151E2B;
  --border:#212B3A; --border-bright:#2E3B4E; --text:#E7ECF3; --text-dim:#8C99AC; --text-faint:#5C6980;
  --signal:#57E1C9; --signal-dim:#2E5952; --amber:#F0B342; --rose:#F0587A; --sage:#7FD99A; --violet:#9C8CF2;
  --risk-low:#7FD99A; --risk-medium:#F0B342; --risk-high:#F08A4B; --risk-critical:#F0587A;
  --font-display:'Space Grotesk', sans-serif; --font-mono:'JetBrains Mono', monospace;
}
*{box-sizing:border-box;}
html{scroll-behavior:smooth;}
body{margin:0; background:var(--bg); color:var(--text); font-family:var(--font-display); line-height:1.55; -webkit-font-smoothing:antialiased;}
a{color:var(--signal);}
.wrap{max-width:960px; margin:0 auto; padding:0 24px; position:relative; z-index:1;}
.topbar{position:sticky; top:0; z-index:50; background:rgba(10,14,19,0.88); backdrop-filter:blur(10px); border-bottom:1px solid var(--border);}
.topbar-inner{max-width:960px; margin:0 auto; padding:14px 24px; display:flex; align-items:center; justify-content:space-between; gap:16px; flex-wrap:wrap;}
.brand{display:flex; align-items:center; gap:10px; font-family:var(--font-mono); font-size:13px; letter-spacing:0.02em; color:var(--text-dim);}
.brand-dot{width:8px; height:8px; border-radius:50%; background:var(--signal); box-shadow:0 0 10px var(--signal);}
.topnav{display:flex; gap:20px; font-family:var(--font-mono); font-size:12.5px;}
.topnav a{color:var(--text-dim); text-decoration:none; border-bottom:1px solid transparent; padding-bottom:2px;}
.topnav a:hover{color:var(--signal); border-color:var(--signal);}
@media (max-width:760px){ .topnav{display:none;} }
.hero{padding:76px 0 48px; border-bottom:1px solid var(--border);}
.hero-eyebrow{font-family:var(--font-mono); font-size:12.5px; color:var(--signal); display:flex; align-items:center; gap:10px; margin-bottom:22px;}
.hero-eyebrow .line{width:28px; height:1px; background:var(--signal-dim);}
h1.hero-title{font-size:clamp(2.1rem, 5vw, 3.4rem); line-height:1.08; margin:0 0 20px; font-weight:600; letter-spacing:-0.01em; max-width:820px;}
.hero-sub{color:var(--text-dim); max-width:640px; font-size:16px; margin-bottom:36px;}
.hero-sub b{color:var(--text); font-weight:500;}
.stat-row{display:grid; grid-template-columns:repeat(4,1fr); gap:1px; background:var(--border); border:1px solid var(--border);}
.stat{background:var(--bg-panel); padding:20px 18px;}
.stat-num{font-family:var(--font-mono); font-size:26px; font-weight:700; color:var(--signal);}
.stat-label{font-size:12.5px; color:var(--text-dim); margin-top:6px;}
@media (max-width:700px){ .stat-row{grid-template-columns:repeat(2,1fr);} }
section{padding:56px 0; border-bottom:1px solid var(--border);}
section:last-of-type{border-bottom:none;}
.section-head{margin-bottom:28px;}
.section-kicker{font-family:var(--font-mono); font-size:12px; color:var(--text-faint); text-transform:uppercase; letter-spacing:0.08em; margin-bottom:10px;}
h2{font-size:1.55rem; font-weight:600; margin:0 0 8px; letter-spacing:-0.01em;}
.section-desc{color:var(--text-dim); max-width:680px; font-size:15px;}
p{color:var(--text-dim); font-size:15px;}
p.lead{color:var(--text); font-size:15.5px;}
.text-block{max-width:700px;}
code{font-family:var(--font-mono); font-size:0.86em; background:var(--bg-panel-raised); padding:1px 6px; border-radius:3px; color:var(--signal);}
.table-wrap{overflow-x:auto; border:1px solid var(--border); border-radius:6px;}
table{width:100%; border-collapse:collapse; font-size:13.5px; min-width:560px;}
thead th{text-align:left; font-family:var(--font-mono); font-size:11.5px; text-transform:uppercase; letter-spacing:0.05em; color:var(--text-faint); padding:12px 14px; background:var(--bg-panel-raised); border-bottom:1px solid var(--border); white-space:nowrap;}
tbody td{padding:11px 14px; border-bottom:1px solid var(--border); color:var(--text-dim); white-space:nowrap;}
tbody tr:last-child td{border-bottom:none;}
tbody tr:hover{background:rgba(87,225,201,0.04);}
td.hl, td.num-hl{color:var(--text); font-family:var(--font-mono);}
td.mono, td.num{font-family:var(--font-mono); color:var(--text);}
tr.winner td{background:rgba(87,225,201,0.06);}
tr.winner td:first-child{box-shadow:inset 3px 0 0 var(--signal);}
.tag{display:inline-block; font-family:var(--font-mono); font-size:11px; padding:2px 8px; border-radius:20px; border:1px solid var(--border-bright); color:var(--text-dim);}
.tag.win{color:var(--signal); border-color:var(--signal-dim); background:rgba(87,225,201,0.08);}
.grid-2{display:grid; grid-template-columns:1fr 1fr; gap:16px;}
.grid-3{display:grid; grid-template-columns:repeat(3,1fr); gap:16px;}
@media (max-width:760px){ .grid-2,.grid-3{grid-template-columns:1fr;} }
.card{background:var(--bg-panel); border:1px solid var(--border); border-radius:6px; padding:22px;}
.card h3{margin:0 0 10px; font-size:15px; font-weight:600;}
.card p{margin:0; font-size:14px;}
.card .metric{font-family:var(--font-mono); font-size:22px; color:var(--text); font-weight:700; margin-bottom:4px;}
.card .metric.good{color:var(--sage);}
.card .metric.warn{color:var(--amber);}
.card .metric.bad{color:var(--rose);}
.bar-row{display:flex; align-items:center; gap:14px; margin-bottom:14px;}
.bar-label{width:180px; font-family:var(--font-mono); font-size:12.5px; color:var(--text-dim); flex-shrink:0;}
.bar-track{flex:1; height:20px; background:var(--bg-panel-raised); border-radius:3px; overflow:hidden; position:relative;}
.bar-fill{height:100%; background:linear-gradient(90deg, var(--signal-dim), var(--signal));}
.bar-fill.alt{background:linear-gradient(90deg, #4A3F73, var(--violet));}
.bar-val{width:70px; text-align:right; font-family:var(--font-mono); font-size:12.5px; color:var(--text); flex-shrink:0;}
.report-card{border:1px solid var(--border-bright); border-radius:8px; overflow:hidden; background:linear-gradient(180deg, var(--bg-panel-raised), var(--bg-panel));}
.report-top{padding:20px 22px; border-bottom:1px solid var(--border); display:flex; justify-content:space-between; align-items:flex-start; gap:16px; flex-wrap:wrap;}
.report-input{font-family:var(--font-mono); font-size:13px; color:var(--text-dim); max-width:560px;}
.report-input .lbl{color:var(--text-faint); font-size:11px; text-transform:uppercase; letter-spacing:0.06em; display:block; margin-bottom:6px;}
.risk-chip{display:flex; align-items:center; gap:8px; font-family:var(--font-mono); font-size:13px; padding:8px 14px; border-radius:20px; background:rgba(240,88,122,0.1); border:1px solid rgba(240,88,122,0.35); color:var(--risk-critical); white-space:nowrap;}
.risk-chip .dot{width:8px; height:8px; border-radius:50%; background:currentColor; box-shadow:0 0 8px currentColor;}
.report-body{padding:4px 22px 22px;}
.report-sub-head{font-family:var(--font-mono); font-size:11px; text-transform:uppercase; letter-spacing:0.07em; color:var(--text-faint); margin:22px 0 10px;}
.evidence-item{padding:12px 14px; border:1px solid var(--border); border-radius:6px; margin-bottom:8px; font-size:13.5px; background:var(--bg-panel);}
.evidence-item .ev-head{display:flex; justify-content:space-between; font-family:var(--font-mono); font-size:11.5px; color:var(--text-faint); margin-bottom:6px;}
.evidence-item .ev-head b{color:var(--signal); font-weight:600;}
.evidence-item p{margin:0; color:var(--text-dim); font-size:13.5px;}
.advice-box{margin-top:20px; padding:16px 18px; border-radius:6px; border:1px solid var(--signal-dim); background:rgba(87,225,201,0.05); font-size:14px; color:var(--text);}
.advice-box b{color:var(--signal);}
.steps{counter-reset:step; margin-top:8px;}
.step{position:relative; padding:0 0 32px 46px; border-left:1px solid var(--border); margin-left:14px;}
.step:last-child{padding-bottom:4px;}
.step::before{counter-increment:step; content:counter(step); position:absolute; left:-14px; top:0; width:28px; height:28px; border-radius:50%; background:var(--bg-panel-raised); border:1px solid var(--signal-dim); color:var(--signal); font-family:var(--font-mono); font-size:13px; font-weight:600; display:flex; align-items:center; justify-content:center;}
.step h3{margin:2px 0 8px; font-size:15.5px; font-weight:600;}
.step p{font-size:14px; margin:0 0 10px;}
pre{background:#0C1219; border:1px solid var(--border); border-radius:6px; padding:16px 18px; overflow-x:auto; font-family:var(--font-mono); font-size:12.5px; color:#C9D6E3; line-height:1.6;}
pre .c1{color:var(--text-faint);} pre .kw{color:var(--violet);} pre .str{color:var(--sage);} pre .fn{color:var(--signal);}
.callout{border-left:3px solid var(--amber); background:rgba(240,179,66,0.06); padding:14px 18px; border-radius:0 6px 6px 0; font-size:14px; color:var(--text-dim); margin:18px 0;}
.callout b{color:var(--amber);}
.callout.good{border-color:var(--sage); background:rgba(127,217,154,0.06);} .callout.good b{color:var(--sage);}
.callout.bad{border-color:var(--rose); background:rgba(240,88,122,0.06);} .callout.bad b{color:var(--rose);}
.arch{display:flex; align-items:stretch; gap:0; flex-wrap:wrap; margin-top:10px;}
.arch-node{flex:1; min-width:130px; background:var(--bg-panel); border:1px solid var(--border); padding:14px; text-align:center; position:relative;}
.arch-node .an-label{font-family:var(--font-mono); font-size:11px; color:var(--text-faint); text-transform:uppercase; letter-spacing:0.05em;}
.arch-node .an-title{font-size:13.5px; font-weight:600; margin-top:6px;}
.arch-arrow{display:flex; align-items:center; justify-content:center; color:var(--text-faint); font-family:var(--font-mono); padding:0 4px; font-size:16px;}
@media (max-width:700px){ .arch{flex-direction:column;} .arch-arrow{transform:rotate(90deg); padding:2px 0;} }
footer{padding:44px 0 60px;}
footer p{font-size:13px; color:var(--text-faint); font-family:var(--font-mono);}
.toc-back{position:fixed; bottom:24px; right:24px; z-index:40;}
.toc-back a{display:flex; align-items:center; justify-content:center; width:42px; height:42px; border-radius:50%; background:var(--bg-panel-raised); border:1px solid var(--border-bright); color:var(--text-dim); text-decoration:none; font-family:var(--font-mono);}
</style>
</head>
<body>
<div class="topbar">
  <div class="topbar-inner">
    <div class="brand"><span class="brand-dot"></span> PHISHING-DETECT // EVAL &amp; DEPLOYMENT REPORT</div>
    <nav class="topnav">
      <a href="#results">Results</a><a href="#demo">Sample Report</a><a href="#robustness">Robustness</a>
      <a href="#signals">Real-World Signals</a><a href="#deploy">Deploy</a><a href="#conclusion">Conclusion</a>
    </nav>
  </div>
</div>
<div class="wrap">
"""

_HTML_HERO = f"""
<section class="hero" style="border-bottom:1px solid var(--border);">
  <div class="hero-eyebrow"><span class="line"></span> RESEARCH REPORT — CLASSICAL ML · TRANSFORMERS · CUSTOM HYBRID</div>
  <h1 class="hero-title">Do modern Transformers actually beat classical NLP at catching phishing — and at what cost?</h1>
  <p class="hero-sub">A leak-safe, statistically-validated comparison of <b>{len(_lb_models)} model families</b> across <b>{HERO_STAT_N} emails</b> from pooled real-world corpora, extended with adversarial stress-testing, social-engineering tactic detection, IOC extraction, and a RAG-grounded explainability agent — packaged into a deployable scoring service. Deployed/selected model: <b>{HERO_STAT_MODEL}</b>.</p>
  <div class="stat-row">
    <div class="stat"><div class="stat-num">{HERO_STAT_F1}</div><div class="stat-label">Best test F1 — {HERO_STAT_MODEL}</div></div>
    <div class="stat"><div class="stat-num">{HERO_STAT_N}</div><div class="stat-label">Emails, pooled sources</div></div>
    <div class="stat"><div class="stat-num">{HERO_STAT_THRESH}</div><div class="stat-label">SOC-optimised threshold</div></div>
    <div class="stat"><div class="stat-num">{HERO_STAT_LATENCY}</div><div class="stat-label">Batched inference / email ({HERO_STAT_MODEL})</div></div>
  </div>
</section>
"""

_HTML_RESULTS = f"""
<section id="results">
  <div class="section-head">
    <div class="section-kicker">01 — Model Leaderboard</div>
    <h2>{len(_lb_models)} finalists, one leak-free test set</h2>
    <p class="section-desc">Tuned classical baselines, fine-tuned Transformers, and a custom Attention-BiGRU fused with hand-engineered features — all scored on an identical, template-grouped held-out split so no near-duplicate email leaks across train/val/test.</p>
  </div>
  <div class="table-wrap">
    <table>
      <thead><tr><th>Model</th><th>Accuracy</th><th>Precision</th><th>Recall</th><th>F1</th><th>ROC-AUC</th><th>Train time</th><th>Params</th></tr></thead>
      <tbody>{LEADERBOARD_ROWS_HTML}</tbody>
    </table>
  </div>
  <div class="callout">{_sig_html}</div>
  <div class="section-kicker" style="margin-top:36px;">Ensembling — a secondary reference, not the headline result</div>
  <div class="grid-2">{ENSEMBLE_CARDS_HTML}</div>
  <div class="section-kicker" style="margin-top:36px;">Cost that actually recurs in production: inference, not training</div>
  <div class="table-wrap">
    <table><thead><tr><th>Model</th><th>Single-item p95</th><th>Batched / item</th><th>Throughput</th><th>On-disk size</th></tr></thead>
      <tbody>{COST_ROWS_HTML}</tbody>
    </table>
  </div>
  <p style="margin-top:12px;">{_deploy_cost_note}</p>
</section>
"""

_HTML_DEMO = f"""
<section id="demo">
  <div class="section-head">
    <div class="section-kicker">02 — Explainable Detection</div>
    <h2>What the system hands an analyst, per email</h2>
    <p class="section-desc">Every flagged email is scored by the deployed model ({_esc(CHAMPION_DISP)}), explained with token attribution where available, checked for indicators of compromise, and matched against a curated social-engineering knowledge base by a retriever.</p>
  </div>
  {DEMO_HTML}
</section>
"""

_HTML_THRESHOLD = f"""
<section>
  <div class="section-head">
    <div class="section-kicker">03 — Operating Point &amp; Errors</div>
    <h2>Tuning the threshold to what a false negative actually costs</h2>
    <p class="section-desc">A missed phishing email is far more expensive to a SOC than a false alarm. Moving off the default 0.5 cut to a cost-minimising threshold changes the trade-off measurably.</p>
  </div>
  <div class="grid-3">{_thresh_cards}</div>
  <div class="section-kicker" style="margin-top:32px;">Error analysis at the tuned threshold ({_esc(CHAMPION_DISP)}, test set)</div>
  <div class="grid-2">{_error_cards}</div>
</section>
"""

_HTML_ROBUSTNESS = f"""
<section id="robustness">
  <div class="section-head">
    <div class="section-kicker">04 — Stress-Testing</div>
    <h2>Where the model actually breaks</h2>
    <p class="section-desc">A held-out F1 score says nothing about a model facing an adversary. This system is tested against source-shift and against text obfuscation designed to evade exactly this kind of classifier.</p>
  </div>
  <div class="section-kicker">Cross-source generalisation (leave-one-source-out, TF-IDF + LogReg)</div>
  <div class="table-wrap">
    <table><thead><tr><th>Held-out source</th><th>n</th><th>F1</th><th>Recall</th><th>Precision</th><th>ROC-AUC</th></tr></thead>
    <tbody>{_cross_source_rows_html}</tbody></table>
  </div>
  <div class="section-kicker" style="margin-top:30px;">Adversarial obfuscation ({_esc(CHAMPION_DISP)})</div>
  <div style="margin-top:14px;">{_obf_bars_html or '<p>Not computed this run.</p>'}</div>
  <p>{_obf_summary}</p>
</section>
"""

_HTML_SIGNALS = f"""
<section id="signals">
  <div class="section-head">
    <div class="section-kicker">05 — Beyond a Label</div>
    <h2>What makes this a SOC tool, not just a classifier</h2>
    <p class="section-desc">A bare PHISHING/legit label is of limited use to an analyst. The pipeline also derives the social-engineering tactic at play and pulls machine-readable indicators of compromise out of every flagged email.</p>
  </div>
  <div class="grid-2">
    <div class="card">
      <h3>Social-engineering tactic detection</h3>
      <p style="margin-bottom:14px;">A weak-supervision lexicon bootstraps labels; a learned multi-label classifier then generalises beyond exact keyword matches, evaluated with per-tactic, validation-tuned thresholds.</p>
      <div class="table-wrap"><table><thead><tr><th>Tactic</th><th>Test F1</th><th>Prevalence</th></tr></thead>
      <tbody>{_tactic_rows_html}</tbody></table></div>
    </div>
    <div class="card">
      <h3>IOC extraction (test set, flagged emails)</h3>
      {_ioc_block}
    </div>
  </div>
</section>
"""

_HTML_DEPLOY = f"""
<section id="deploy">
  <div class="section-head">
    <div class="section-kicker">06 — Deployment Guide</div>
    <h2>Running this as a real scoring service</h2>
    <p class="section-desc">{_deploy_cost_note} Swapping in the {_esc(CHAMPION_DISP)} checkpoint only requires changing the model-loading block below.</p>
  </div>
  <div class="steps">
    <div class="step">
      <h3>Export the trained artifacts</h3>
      <p>Section 24.1 serialises everything the reference service needs: the fitted TF-IDF vectorizer, the logistic regression model, and a small config file carrying the SOC-tuned decision threshold.</p>
      <pre><span class="c1"># produced by the notebook into ./deployment/</span>
tfidf_vectorizer.joblib
logreg_model.joblib
deploy_config.json   <span class="c1"># {{"threshold": ..., "model": "TFIDF_LogReg"}}</span></pre>
    </div>
    <div class="step">
      <h3>Score emails with the FastAPI service</h3>
      <p>A minimal, self-contained scoring API. It loads the vectorizer and model once at startup and exposes a single <code>/predict</code> endpoint.</p>
      <pre><span class="kw">import</span> json, re
<span class="kw">from</span> pathlib <span class="kw">import</span> Path
<span class="kw">import</span> joblib
<span class="kw">from</span> fastapi <span class="kw">import</span> FastAPI
<span class="kw">from</span> pydantic <span class="kw">import</span> BaseModel

vectorizer = joblib.load(<span class="str">"tfidf_vectorizer.joblib"</span>)
model      = joblib.load(<span class="str">"logreg_model.joblib"</span>)
THRESHOLD  = json.loads(Path(<span class="str">"deploy_config.json"</span>).read_text())[<span class="str">"threshold"</span>]

app = FastAPI(title=<span class="str">"Phishing Detection API"</span>)

<span class="kw">class</span> EmailIn(BaseModel):
    text: str

<span class="kw">@app.post</span>(<span class="str">"/predict"</span>)
<span class="kw">def</span> <span class="fn">predict</span>(payload: EmailIn):
    cleaned = clean_text(payload.text)
    proba = <span class="fn">float</span>(model.predict_proba(vectorizer.transform([cleaned]))[0, 1])
    <span class="kw">return</span> {{
        <span class="str">"probability"</span>: proba,
        <span class="str">"label"</span>: <span class="str">"phishing"</span> <span class="kw">if</span> proba >= THRESHOLD <span class="kw">else</span> <span class="str">"legit"</span>,
        <span class="str">"threshold"</span>: THRESHOLD,
    }}</pre>
      <p>Run it locally with:</p>
      <pre>uvicorn app:app --host 0.0.0.0 --port 8000</pre>
    </div>
    <div class="step">
      <h3>Containerise it</h3>
      <p>A slim Python image with only the dependencies the service actually needs — no GPU base image required.</p>
      <pre>FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py tfidf_vectorizer.joblib logreg_model.joblib deploy_config.json ./
EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]</pre>
      <pre>docker build -t phishing-api .
docker run -p 8000:8000 phishing-api</pre>
    </div>
    <div class="step">
      <h3>Give analysts a UI</h3>
      <p>A lightweight Streamlit demo that loads the model directly, or points at the running FastAPI service for an end-to-end demo of the deployed path.</p>
      <pre>pip install streamlit
streamlit run streamlit_app.py</pre>
    </div>
    <div class="step">
      <h3>Wire in the explainability agent (optional)</h3>
      <p>For SOC-facing deployments, set <code>ANTHROPIC_API_KEY</code> and point the LangGraph agent at the same knowledge base used above to get a full grounded explanation report — token attribution, tactics, IOCs, retrieved evidence, and a hallucination-checked narrative — per flagged email, in the same shape as the sample report above.</p>
    </div>
  </div>
</section>
"""

_HTML_CONCLUSION = f"""
<section id="conclusion">
  <div class="section-head">
    <div class="section-kicker">07 — Conclusion</div>
    <h2>Answering the research question</h2>
  </div>
  <p class="text-block lead">{CONCLUSION_HTML}</p>
  <p class="text-block">This is the version of the project meant to be shipped, not just benchmarked: a cost-aware operating threshold, honest error analysis, adversarial and cross-source stress tests, IOC and tactic extraction for downstream SOC tooling, a retrieval-grounded explanation layer, and a containerised scoring API.</p>
</section>
</div>
<div class="toc-back"><a href="#" title="Back to top">↑</a></div>
<footer><div class="wrap"><p>DATA PROVENANCE: {_esc(globals().get('DATA_SOURCE_NOTE', 'see Section 2'))} · all figures read directly from this run's own RESULTS/BEST_MODEL_NAME — none hand-typed.</p></div></footer>
</body>
</html>
"""

_HTML_REPORT = (_HTML_HEAD + _HTML_HERO + _HTML_RESULTS + _HTML_DEMO + _HTML_THRESHOLD
                + _HTML_ROBUSTNESS + _HTML_SIGNALS + _HTML_DEPLOY + _HTML_CONCLUSION)

_report_path = REPORT_DIR / "phishing_detection_report.html"
_report_path.write_text(_HTML_REPORT, encoding="utf-8")
print(f"Wrote standalone HTML report to: {_report_path}")
print(f"File size: {_report_path.stat().st_size / 1024:.1f} KB")
print(f"Report describes the champion/winning model: {CHAMPION_DISP}")
print("Open it directly in a browser -- no server, no dependencies.")


Report data assembled for champion model: RoBERTa-base
Wrote standalone HTML report to: /kaggle/working/report/phishing_detection_report.html
File size: 32.1 KB
Report describes the champion/winning model: RoBERTa-base
Open it directly in a browser -- no server, no dependencies.


<a id="sec28"></a>

## 28. Interactive Dashboard — ipywidgets Phishing Risk Report

A colourful, clinical-dashboard-style report for a **single email**, built with
`ipywidgets` so it runs live inside this notebook (pick a demo email or paste
your own, click **Run Detection**, and the report renders below the button).

It reuses the pipeline already built above rather than re-implementing anything:

- **Prediction + calibrated risk** → `run_soc_agent()` (Section 25.4), which
  chains the champion model, the tactic/IOC detectors, the risk-scoring
  formula, and the RAG retriever into one call.
- **Top contributing factors** → the same token-attribution list
  (`shap_equivalent_tokens`) the agent already computes in `_n_tokens`.
- **Retrieved evidence & validation-agent verdict** → `rag_sources` +
  `validation` from the agent's final report, shown the same way Section 22's
  `generate_explanation_report()` prints them, just styled as cards.

If the full pipeline hasn't been run in this session (e.g. `run_soc_agent` is
undefined because Sections 1–25 above haven't executed, or you're testing this
cell in isolation), the widget automatically falls back to a lightweight,
self-contained heuristic analyzer so the UI is still demonstrable — the report
banner says so explicitly whenever that happens, so a fallback result is never
mistaken for a real model verdict.


In [1]:
import re as _dash_re
import html as _dash_html
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ---------------------------------------------------------------------------
# 28.1  Fallback analyzer -- used ONLY if the full pipeline (Section 25's
# run_soc_agent) isn't available in this kernel session, so the dashboard is
# still demonstrable on its own. Mirrors the shape of `final_report` returned
# by run_soc_agent() so _dashboard_html() below never needs to know which
# path produced it. NEVER silently presented as a real model verdict -- the
# banner text says "FALLBACK / HEURISTIC MODE" whenever this path is used.
# ---------------------------------------------------------------------------
_FALLBACK_URGENCY_WORDS = ["urgent", "verify", "suspend", "immediately", "password",
                           "click", "confirm", "account", "limited", "expire",
                           "act now", "security alert", "winner", "congratulations"]
_FALLBACK_SCAM_LEXICONS = {
    "credential_harvesting": ["verify your account", "confirm your password", "sign in to", "login to your account"],
    "business_email_compromise": ["wire transfer", "gift card", "purchase order", "banking details"],
    "malware_delivery": ["enable macros", "open the attachment", "download the file", ".exe"],
    "advance_fee": ["inheritance", "lottery", "next of kin", "processing fee"],
}
_FALLBACK_KB = [
    ("kb_urgency", "Urgency / time-pressure social engineering",
     "Urgency-based social engineering pressures the recipient to act within an artificially "
     "short deadline, suppressing careful verification.", "SOC playbook"),
    ("kb_url_ioc", "URL IOC handling guidance",
     "Extracted URLs are IOCs: never click directly, check domain registration age, and block "
     "confirmed-malicious domains at the perimeter.", "SOC playbook / CISA phishing guidance"),
    ("kb_credential_harvesting", "Credential harvesting pattern",
     "Directs the recipient to a fake login page mimicking a legitimate service. Never enter "
     "credentials via an unsolicited email link.", "MITRE ATT&CK T1566.002"),
    ("kb_false_positive", "Legitimate transactional email",
     "Receipts, password-reset confirmations the user themselves triggered, and calendar invites "
     "can superficially resemble phishing on length/link-count features alone.", "SOC playbook"),
]

def _fallback_extract_iocs(text):
    urls = _dash_re.findall(r"https?://[^\s\"'<>]+|www\.[^\s\"'<>]+", text, _dash_re.I)
    emails = _dash_re.findall(r"[\w.+-]+@[\w-]+\.[\w.-]+", text)
    ips = _dash_re.findall(r"\b(?:(?:25[0-5]|2[0-4]\d|[01]?\d?\d)\.){3}(?:25[0-5]|2[0-4]\d|[01]?\d?\d)\b", text)
    return {"urls": urls, "emails": emails, "ips": ips}

def _fallback_analyze(email_text):
    text_lower = email_text.lower()
    hits = [w for w in _FALLBACK_URGENCY_WORDS if w in text_lower]
    proba = min(0.97, 0.12 + 0.14 * len(hits))
    prediction = "phishing" if proba >= 0.5 else "legitimate"
    confidence = proba if prediction == "phishing" else 1 - proba

    iocs = _fallback_extract_iocs(email_text)
    ioc_count = len(iocs["urls"]) + len(iocs["emails"]) + len(iocs["ips"])

    tactics = [w.replace(" ", "_") for w in hits[:5]]

    scam_hits = {k: [p for p in v if p in text_lower] for k, v in _FALLBACK_SCAM_LEXICONS.items()}
    scam_hits = {k: v for k, v in scam_hits.items() if v}
    scam_type = max(scam_hits.items(), key=lambda kv: len(kv[1]))[0] if scam_hits else "unclassified"

    ml_score = confidence if prediction == "phishing" else 1 - confidence
    ioc_boost = min(ioc_count, 5) * 4
    tactic_boost = min(len(tactics), 4) * 2.5
    scam_boost = 5 if scam_type != "unclassified" else 0
    score = max(0.0, min(100.0, ml_score * 70 + ioc_boost + tactic_boost + scam_boost))
    tier = "critical" if score >= 75 else "high" if score >= 50 else "medium" if score >= 25 else "low"

    top_tokens = [(w, round(0.05 + 0.03 * i, 3)) for i, w in enumerate(hits[:8][::-1])]

    query_words = set(text_lower.split())
    def _kb_score(kb_text):
        return len(query_words & set(kb_text.lower().split()))
    retrieved = sorted(_FALLBACK_KB, key=lambda d: -_kb_score(d[2]))[:3]
    rag_sources = [{"rank": i + 1, "score": round(0.35 - 0.08 * i, 2), "title": t, "source": s, "id": kid}
                    for i, (kid, t, txt, s) in enumerate(retrieved)]

    explanation = (
        "[FALLBACK HEURISTIC MODE -- full pipeline (run_soc_agent) not found in this session; "
        "this is a lexicon-based estimate, not the trained model's output] "
        f"The message was flagged '{prediction}' based on {len(hits)} urgency/credential-lexicon "
        f"keyword hit(s) and {ioc_count} extracted indicator(s) of compromise. "
        + ("Recommended: quarantine, extract IOCs, and verify via a separate known-good channel."
           if tier in ("high", "critical") else
           "Recommended: log the disposition and sample for periodic analyst review.")
    )

    return {
        "prediction": prediction, "confidence": confidence,
        "risk_tier": tier, "risk_score": round(score, 1),
        "risk_components": {"ml": round(ml_score * 70, 1), "ioc": round(ioc_boost, 1),
                             "tactic": round(tactic_boost, 1), "scam_type": round(scam_boost, 1)},
        "scam_type": scam_type, "tactics_detected": tactics,
        "tactics_detection_method": "heuristic_lexicon (fallback mode)",
        "shap_equivalent_tokens": [{"token": t, "attribution": v} for t, v in top_tokens],
        "iocs": iocs, "iocs_detection_method": "heuristic_regex (fallback mode)",
        "rag_sources": rag_sources,
        "explanation": explanation, "llm_model_used": "FALLBACK_HEURISTIC_NOT_A_MODEL",
        "validation": {"passed": True, "mean_groundedness": 0.5, "n_sentences": 1,
                        "verdict_contradiction_detected": False},
        "recommended_actions_source": "kb_actions_high_risk" if tier in ("high", "critical") else "kb_actions_low_risk",
        "node_timings_seconds": {}, "total_latency_seconds": 0.0,
        "_fallback_mode": True,
    }

def _analyze_email(email_text):
    if "run_soc_agent" in globals():
        report = run_soc_agent(email_text)
        report.setdefault("_fallback_mode", False)
        return report
    return _fallback_analyze(email_text)

# ---------------------------------------------------------------------------
# 28.2  Styling helpers
# ---------------------------------------------------------------------------
_TIER_STYLE = {
    "critical": dict(bg="#fdecea", border="#c0392b", text="#a93226", label="CRITICAL"),
    "high":     dict(bg="#fdecea", border="#e67e22", text="#c0392b", label="HIGH"),
    "medium":   dict(bg="#fff8e1", border="#f0ad4e", text="#9a6d00", label="MODERATE"),
    "low":      dict(bg="#eafaf1", border="#27ae60", text="#1e8449", label="LOW"),
}

def _pred_badge(prediction):
    if prediction == "phishing":
        return '<span style="color:#c0392b;font-weight:700;">&#9650; PHISHING</span>'
    return '<span style="color:#1e8449;font-weight:700;">&#10003; LEGITIMATE</span>'

def _esc(s):
    return _dash_html.escape(str(s))

def _pill(text, bg="#eef1fb", color="#3b4b8c", border="#c7cff0"):
    return (f'<span style="display:inline-block;padding:1px 8px;border-radius:10px;'
            f'font-size:11px;font-weight:600;background:{bg};color:{color};'
            f'border:1px solid {border};margin:1px 3px 1px 0;">{_esc(text)}</span>')

def _section_title(n, title):
    return (f'<div style="font-size:13px;font-weight:700;letter-spacing:.3px;color:#1a2456;'
            f'text-transform:uppercase;margin:22px 0 8px 0;border-bottom:1px solid #dfe3ee;'
            f'padding-bottom:5px;">{n}. {_esc(title)}</div>')

def _card_open(border="#dfe3ee"):
    return (f'<div style="border:1px solid {border};border-left:4px solid {border};'
            f'border-radius:6px;padding:14px 16px;margin-bottom:12px;background:#fff;">')

def _card_close():
    return '</div>'

# ---------------------------------------------------------------------------
# 28.3  Main HTML builder -- mirrors the clinical-risk-dashboard layout:
# header -> headline alert -> overview table -> detail card -> retrieved
# evidence / validation-agent verdicts.
# ---------------------------------------------------------------------------
def _dashboard_html(report, email_text):
    tier = report.get("risk_tier", "low")
    ts = _TIER_STYLE.get(tier, _TIER_STYLE["low"])
    prediction = report.get("prediction", "legitimate")
    confidence = report.get("confidence", 0.0)
    score = report.get("risk_score", 0.0)
    scam_type = report.get("scam_type", "unclassified")
    tactics = report.get("tactics_detected", [])
    tokens = report.get("shap_equivalent_tokens", [])
    iocs = report.get("iocs", {"urls": [], "emails": [], "ips": []})
    ioc_count = len(iocs.get("urls", [])) + len(iocs.get("emails", [])) + len(iocs.get("ips", []))
    rag_sources = report.get("rag_sources", [])
    validation = report.get("validation", {})
    explanation = report.get("explanation", "")
    is_fallback = report.get("_fallback_mode", False)
    model_used = report.get("llm_model_used", "n/a")
    recs_hint = ("Quarantine the message, extract & block IOCs, verify via a separate known-good "
                 "channel, notify affected users." if tier in ("high", "critical") else
                 "Log the disposition; sample for periodic analyst review.")

    max_attr = max([abs(t["attribution"]) for t in tokens], default=1.0) or 1.0

    # ---- header -----------------------------------------------------------
    html = []
    html.append(
        '<div style="font-family:-apple-system,Segoe UI,Roboto,Helvetica,Arial,sans-serif;'
        'max-width:880px;margin:0 auto;color:#1a1a2e;line-height:1.45;">'
    )
    html.append(
        '<div style="display:flex;align-items:center;gap:10px;border-bottom:2px solid #1a2456;'
        'padding-bottom:10px;margin-bottom:2px;">'
        '<span style="font-size:24px;">&#127907;</span>'
        '<div><div style="font-size:19px;font-weight:800;color:#1a2456;">'
        'Phishing Email Risk Dashboard</div>'
        '<div style="font-size:11.5px;color:#8892a6;">SOC Analyst Report &mdash; '
        'LangGraph Agent (research demo, not production use)</div></div></div>'
    )

    if is_fallback:
        html.append(
            '<div style="margin-top:10px;padding:8px 12px;background:#fef6e0;border:1px solid '
            '#f0ad4e;border-radius:6px;font-size:12px;color:#8a6100;">'
            '&#9888; <b>FALLBACK / HEURISTIC MODE</b> &mdash; the full trained pipeline '
            '(<code>run_soc_agent</code>) was not found in this kernel session, so this report '
            'uses a lightweight lexicon-based estimate for demonstration only. Run Sections 1&ndash;25 '
            'above first for the real model verdict.</div>'
        )

    # ---- headline alert banner --------------------------------------------
    html.append(
        f'<div style="margin-top:14px;padding:12px 16px;background:{ts["bg"]};'
        f'border:1px solid {ts["border"]};border-radius:8px;">'
        f'<div style="font-size:14.5px;font-weight:700;color:{ts["text"]};">'
        f'&#9888; Calibrated risk is <u>{ts["label"]}</u> at {score:.1f}/100, '
        f'prediction: {_pred_badge(prediction)} (confidence {confidence:.1%}).</div>'
        f'<div style="font-size:12.5px;color:#4a4a4a;margin-top:5px;">'
        f'<b>Recommendation:</b> {_esc(recs_hint)}</div>'
        f'</div>'
    )

    # ---- Section 1: overview table -----------------------------------------
    html.append(_section_title(1, "Detection Overview"))
    rows = [
        ("Prediction", _pred_badge(prediction)),
        ("Confidence", f"{confidence:.1%}"),
        ("Calibrated risk score", f"{score:.1f} / 100"),
        ("Risk category", _pill(ts["label"], bg=ts["bg"], color=ts["text"], border=ts["border"])),
        ("Likely scam type", _esc(scam_type.replace("_", " "))),
        ("Tactics detected", str(len(tactics))),
        ("IOCs extracted", str(ioc_count)),
        ("Model", _esc(model_used if not is_fallback else "heuristic fallback")),
    ]
    html.append('<table style="width:100%;border-collapse:collapse;font-size:13px;">')
    for i, (k, v) in enumerate(rows):
        bg = "#fbfbfe" if i % 2 == 0 else "#ffffff"
        html.append(
            f'<tr style="background:{bg};"><td style="padding:6px 10px;color:#6b7280;'
            f'width:220px;border-bottom:1px solid #eef0f6;">{_esc(k)}</td>'
            f'<td style="padding:6px 10px;font-weight:600;border-bottom:1px solid #eef0f6;">{v}</td></tr>'
        )
    html.append('</table>')

    # ---- Section 2: email-level detail --------------------------------------
    html.append(_section_title(2, "Email-Level Detail"))
    html.append(_card_open(border=ts["border"]))
    html.append(
        f'<div style="display:flex;justify-content:space-between;align-items:flex-start;">'
        f'<div><div style="font-size:14px;font-weight:700;color:#1a2456;">Model Output</div>'
        f'<div style="font-size:12px;color:#6b7280;margin-top:2px;">Prediction: {_pred_badge(prediction)}'
        f' &nbsp;|&nbsp; Confidence: {confidence:.1%}</div></div>'
        f'<span style="padding:3px 10px;border-radius:12px;font-size:11px;font-weight:700;'
        f'background:{ts["bg"]};color:{ts["text"]};border:1px solid {ts["border"]};">{ts["label"]}</span>'
        f'</div>'
    )
    # risk bar
    bar_color = ts["border"]
    html.append(
        f'<div style="margin:10px 0 4px 0;font-size:11.5px;color:#6b7280;">Calibrated risk</div>'
        f'<div style="background:#eef0f6;border-radius:6px;height:10px;width:100%;overflow:hidden;">'
        f'<div style="background:{bar_color};height:10px;width:{min(score,100):.1f}%;"></div></div>'
        f'<div style="font-size:11px;color:#6b7280;margin-top:2px;">{score:.1f}%</div>'
    )

    if tokens:
        html.append('<div style="font-size:12.5px;font-weight:700;color:#1a2456;margin-top:14px;">'
                     'Top Contributing Factors (token attribution)</div>')
        html.append('<ul style="margin:6px 0 0 0;padding-left:18px;font-size:12.5px;">')
        for t in sorted(tokens, key=lambda x: -abs(x["attribution"]))[:8]:
            frac = abs(t["attribution"]) / max_attr
            bar_c = "#c0392b" if t["attribution"] > 0 else "#2e7d32"
            html.append(
                f'<li style="margin-bottom:4px;">'
                f'<code style="background:#f3f4fa;padding:1px 5px;border-radius:4px;">{_esc(t["token"])}</code> '
                f'{_pill("attribution " + format(t["attribution"], "+.3f"), bg="#eef1fb", color="#3b4b8c", border="#c7cff0")}'
                f'<div style="background:#eef0f6;border-radius:4px;height:5px;width:160px;display:inline-block;'
                f'vertical-align:middle;margin-left:6px;overflow:hidden;">'
                f'<div style="background:{bar_c};height:5px;width:{frac*100:.0f}%;"></div></div>'
                f'</li>'
            )
        html.append('</ul>')
    else:
        html.append('<div style="font-size:12px;color:#9aa0ab;margin-top:10px;">No token attributions available.</div>')

    if tactics:
        html.append('<div style="font-size:12.5px;font-weight:700;color:#1a2456;margin-top:14px;">'
                     'Detected Social-Engineering Tactics</div><div style="margin-top:5px;">')
        for tac in tactics:
            html.append(_pill(tac.replace("_", " "), bg="#fdecea", color="#a93226", border="#f3c5be"))
        html.append('</div>')

    html.append('<div style="font-size:12.5px;font-weight:700;color:#1a2456;margin-top:14px;">'
                 'Extracted Indicators of Compromise (IOCs)</div>')
    if ioc_count:
        html.append('<table style="width:100%;border-collapse:collapse;font-size:12px;margin-top:5px;">')
        for kind, values in [("URL", iocs.get("urls", [])), ("Email", iocs.get("emails", [])),
                              ("IP", iocs.get("ips", []))]:
            for v in values:
                html.append(
                    f'<tr><td style="padding:4px 8px;color:#6b7280;width:70px;border-bottom:1px solid #f0f1f6;">{kind}</td>'
                    f'<td style="padding:4px 8px;border-bottom:1px solid #f0f1f6;"><code>{_esc(v)}</code></td></tr>'
                )
        html.append('</table>')
    else:
        html.append('<div style="font-size:12px;color:#9aa0ab;margin-top:4px;">None found in this message.</div>')

    html.append('<div style="font-size:11.5px;font-weight:700;color:#1a2456;margin-top:14px;">Email excerpt</div>')
    excerpt = _esc(email_text[:400]) + ("&hellip;" if len(email_text) > 400 else "")
    html.append(f'<div style="font-family:monospace;font-size:11.5px;background:#f7f8fc;'
                f'border:1px solid #eef0f6;border-radius:5px;padding:8px 10px;margin-top:4px;'
                f'white-space:pre-wrap;color:#3a3f4b;">{excerpt}</div>')
    html.append(_card_close())

    # ---- Section 3: retrieved evidence & validation-agent verdict -----------
    html.append(_section_title(3, "Retrieved Evidence & Validation-Agent Verdict"))
    if rag_sources:
        for src in rag_sources:
            html.append(_card_open(border="#c7cff0"))
            html.append(
                f'<div style="display:flex;justify-content:space-between;align-items:center;">'
                f'<div style="font-size:12.5px;font-weight:700;color:#1a2456;">{_esc(src.get("title",""))}</div>'
                f'{_pill("relevance " + format(src.get("score",0.0), ".2f"))}'
                f'</div>'
                f'<div style="font-size:11px;color:#8892a6;margin-top:2px;">source: {_esc(src.get("source",""))} '
                f'&nbsp;|&nbsp; id: {_esc(src.get("id",""))}</div>'
            )
            html.append(_card_close())
    else:
        html.append('<div style="font-size:12px;color:#9aa0ab;">No supporting evidence retrieved.</div>')

    v_ok = validation.get("passed", False)
    v_bg, v_border, v_text = ("#eafaf1", "#27ae60", "#1e8449") if v_ok else ("#fdecea", "#c0392b", "#a93226")
    html.append(
        f'<div style="padding:10px 14px;background:{v_bg};border:1px solid {v_border};'
        f'border-radius:6px;margin-top:6px;">'
        f'<div style="font-size:12.5px;font-weight:700;color:{v_text};">'
        f'{"&#10003; VALIDATION PASSED" if v_ok else "&#9888; VALIDATION FLAGGED"} '
        f'&mdash; mean groundedness {validation.get("mean_groundedness", 0):.2f}</div>'
        f'<div style="font-size:12px;color:#4a4a4a;margin-top:6px;">{_esc(explanation)}</div>'
        f'</div>'
    )

    html.append('</div>')
    return "".join(html)

# ---------------------------------------------------------------------------
# 28.4  ipywidgets UI
# ---------------------------------------------------------------------------
_demo_emails = globals().get("DEMO_EMAILS", [
    "Urgent: your account will be suspended in 24 hours. Click http://verify-account-now.example.com "
    "to confirm your password immediately or access will be permanently terminated.",
    "Hi team, attaching the Q3 report ahead of tomorrow's review meeting. Let me know if you have "
    "any questions before then. Thanks!",
    "URGENT: This is the CEO. I need you to purchase 5 Amazon gift cards worth $200 each right now "
    "and send me the codes. Keep this between us for now.",
])

_email_options = [(f"Demo {i+1}: {e[:60]}{'...' if len(e) > 60 else ''}", e)
                   for i, e in enumerate(_demo_emails)]
_email_options.append(("Custom email (type/paste below)", "__custom__"))

_dropdown = widgets.Dropdown(
    options=_email_options, description="Sample:",
    style={"description_width": "60px"}, layout=widgets.Layout(width="640px"))
_custom_box = widgets.Textarea(
    placeholder="Paste an email to analyze...",
    layout=widgets.Layout(width="640px", height="90px", display="none"))
_run_btn = widgets.Button(description="Run Detection", icon="search",
                           button_style="danger", layout=widgets.Layout(width="160px"))
# Rationale for this logic: see Appendix A (change log).
_report_html = widgets.HTML(value='<i style="color:#9aa0ab;">'
                             'Select a sample or paste an email, then click Run Detection.</i>')
_status = widgets.HTML(value="")
_debug_out = widgets.Output()

def _on_dropdown_change(change):
    _custom_box.layout.display = "block" if change["new"] == "__custom__" else "none"

_dropdown.observe(_on_dropdown_change, names="value")

def _on_run_clicked(_):
    _debug_out.clear_output()
    _status.value = '<i style="color:#8892a6;">Running detection&hellip;</i>'
    try:
        email_text = (_custom_box.value.strip() if _dropdown.value == "__custom__"
                      else _dropdown.value)
        if not email_text:
            _status.value = ""
            _report_html.value = ('<i style="color:#9aa0ab;">Please select a sample or paste '
                                   'an email first.</i>')
            return
        report = _analyze_email(email_text)
        _report_html.value = _dashboard_html(report, email_text)
        _status.value = ""
    except Exception:
        import traceback
        _status.value = ('<div style="color:#c0392b;font-weight:700;">'
                          '&#9888; Something went wrong -- see traceback below.</div>')
        with _debug_out:
            traceback.print_exc()

_run_btn.on_click(_on_run_clicked)

phishing_dashboard_ui = widgets.VBox([
    widgets.HTML('<div style="font-size:13px;font-weight:700;color:#1a2456;margin-bottom:4px;">'
                 'Select or paste an email to analyze</div>'),
    _dropdown, _custom_box, _run_btn, _status, _report_html, _debug_out,
])
display(phishing_dashboard_ui)


<a id="appendixA"></a>

## Appendix A: Change Log

This appendix replaces the earlier "Development & Revision Log" and the long *Bug fix* comments that used to sit in code cells (they are preserved, unedited, in the collapsed block at the end). It has no bearing on the results.

**Correctness fixes made while building the pipeline (earlier drafts)**
- Transformer checkpoints could not be reloaded after training, so no Transformer could enter the ensemble, calibration, robustness tests or best-model selection. Winning checkpoints are now recorded (`TRANSFORMER_CKPT_MAP`) and reloaded on demand.
- The best-model lookup used fragile substring matching; replaced by one direct lookup.
- Near-duplicates were only diagnosed, never grouped before splitting; now clustered with `connected_components` over a cosine-similarity graph and used as the `GroupShuffleSplit` key.
- The adversarial stress test had no Transformer or ensemble branch; both were added.
- The stacked ensemble originally used validation predictions; a genuine 5-fold out-of-fold stack was added for the classical pair (12.2) and the validation-based stack kept as a reference (12.3).
- A name clash between Hugging Face's `Dataset` and PyTorch's `Dataset` crashed Section 10 when Transformers were enabled; the Hugging Face import is now aliased.

**Revision after review (this version)**
- **Selection protocol:** best model (14), calibration method (15.2) and the significance-test pair (13) now use validation data only; test metrics are reported after selection.
- **Dataset artefacts:** reading note added to 3.4; source-artefact masking test (20.2); limitations rewritten to say the score is inflated by old, source-specific corpora.
- **Stale text:** ablation interpretations (19.1, 19.2) rewritten from the actual outputs; abstract ROC-AUC corrected to 0.9997; conclusion generator fixed (ensemble compared against its own base models, missing model size no longer printed as `nan`).
- **Unfinished items:** `captum` install cell added and the Integrated Gradients claim qualified; normalisation defence (21.3) added; agent/RAG evaluation status stated; title-page fields still to be filled in by the author.
- **Threshold:** boundary problem explained; extended grid and validation-chosen operating points added (15.3b).
- **Structure:** inline *Bug fix* comments moved here; Limitations and Future Work added (26.1); Section 26 text corrected.

<details><summary>Original inline "Bug fix" comments (moved from code cells, unedited)</summary>

**Original cell 38:**

```text
--- Full clustering pass (bug fix: this used to be a TODO comment, not code).
Builds a sparse cosine-similarity radius graph over the WHOLE pooled dataset
(bounded so it stays tractable: falls back to exact-duplicate-only grouping
above ~120k rows, which is documented rather than silently degraded) and
merges any pair above NEAR_DUP_THRESHOLD with connected_components -- the
standard way to turn a pairwise "is-near-duplicate-of" graph into disjoint
template clusters (a union-find pass under the hood).
```

**Original cell 40:**

```text
Bug fix / robustness: GroupShuffleSplit shuffles whole GROUPS, not rows. When there
are few groups relative to the split fractions (e.g. this notebook's synthetic
fallback data -- Section 2 -- collapses to only ~10 template clusters, one per
hand-written template), a single fixed-seed split can by chance put every group of
one class into val or test, leaving that split single-class. That used to crash
every downstream classification_report/LogisticRegression.fit call with a confusing
"only one class present" error far away from the real cause. We now retry the group
split over a small, deterministic sequence of seeds (derived from SEED, so still
fully reproducible) until every split contains both classes, and only fall back to
a plain stratified (non-grouped) split -- clearly flagged -- if that's structurally
impossible (too few groups of the minority class to go around). This never affects
real-sized datasets, where there are far more groups than needed.
```

**Original cell 53:**

```text
Bug fix: earlier versions of this notebook trained each Transformer (Section 9)
via a HuggingFace Trainer, computed aggregate val/test metrics, and then let the
Trainer/model object fall out of scope. That meant get_predictions() could never
reconstruct a Transformer's probabilities later, so no Transformer could ever
enter the ensemble, calibration, threshold optimisation, cross-source test,
adversarial test, or BEST_MODEL selection -- even if it had the best validation
F1. Fix: Trainer already checkpoints to disk each epoch (TrainingArguments'
save_strategy="epoch"); we just need to remember WHICH checkpoint directory
holds the winning hyperparameter trial for each Transformer, and reload it
on demand (Section 12.1) via AutoModel.from_pretrained(that_dir). This dict is
populated by Sections 9.1-9.5.
```

**Original cell 53:**

```text
Naming-consistency fix: internal keys stay as the historical mixed-case identifiers
used throughout this notebook (changing them everywhere would be a large, risky
find/replace across 150+ cells); instead every table/plot that prints a model name
now goes through this single lookup so the *displayed* names are consistent.
```

**Original cell 53:**

```text
Bug fix: pass explicit labels=[0, 1] so this never crashes when a
(possibly degenerate/undertrained) model predicts only one class --
classification_report would otherwise raise ValueError because the
number of classes actually seen no longer matches len(target_names).
```

**Original cell 69:**

```text
Bug fix: the previous version of this cell only checked package installation, so on
a runtime with the libraries installed but no outbound route to huggingface.co it
would proceed straight into from_pretrained() calls and fail deep inside
Trainer.train() with an unhelpful low-level connection traceback instead of a clear,
early message.
```

**Original cell 72:**

```text
^ bug fix: needed so get_predictions("distilbert", ...) can reload the actual
  winning checkpoint later (Section 12.1) instead of raising
  NotImplementedError forever, which silently excluded every Transformer
  from BEST_MODEL selection no matter how well it scored.
```

**Original cell 74:**

```text
^ bug fix: needed so get_predictions("roberta", ...) can reload the actual
  winning checkpoint later (Section 12.1) instead of raising
  NotImplementedError forever, which silently excluded every Transformer
  from BEST_MODEL selection no matter how well it scored.
```

**Original cell 76:**

```text
^ bug fix: needed so get_predictions("deberta_v3", ...) can reload the actual
  winning checkpoint later (Section 12.1) instead of raising
  NotImplementedError forever, which silently excluded every Transformer
  from BEST_MODEL selection no matter how well it scored.
```

**Original cell 87:**

```text
Bug fix: this used to always print "Final 5 models" even when RUN_TRANSFORMERS=False
left TOP2_TRANSFORMERS empty (so FINALISTS has 3 entries, not 5) -- count dynamically.
```

**Original cell 101:**

```text
Bug fix: the previous version of this cell picked BEST_MODEL_NAME from
ci_df, then tried up to 3 increasingly fragile fallback strategies to find
a matching row elsewhere -- including case-insensitive SUBSTRING matching
(`BEST_MODEL_NAME.lower() in idx.lower()`), which is exactly the kind of
logic that quietly matches the wrong row (e.g. "roberta" substring-matching
"distilbert_lora" style names) instead of failing loudly. It existed because
an earlier bug (Fix 4, above) meant Transformers could never actually reach
ci_df, so this cell was patched defensively around a symptom rather than the
cause. With that cause fixed, RESULTS/ci_df/results_df all share one
consistent naming convention (f"{model_key}_test"), so a single direct
lookup is correct and any failure here is a real bug worth seeing, not
something to paper over with fuzzy matching.
Bug fix: results_df (built in Section 11) is a one-time snapshot of RESULTS
taken *before* Section 12.2/12.3 add the ensemble rows (e.g. "EnsembleStack_test",
"EnsembleStackOOF_classical_test") to the RESULTS dict. Looking the winner up in
that stale snapshot is exactly the kind of desync this cell's own comment above
warns about -- so we rebuild results_df from the current RESULTS dict here rather
than trusting the Section-11 copy to still be complete.
```

**Original cell 101:**

```text
Bug fix: results_df (built in Section 11) is a one-time snapshot of RESULTS
taken *before* Section 12.2/12.3 add the ensemble rows (e.g. "EnsembleStack_test",
"EnsembleStackOOF_classical_test") to the RESULTS dict. Looking the winner up in
that stale snapshot is exactly the kind of desync this cell's own comment above
warns about -- so we rebuild results_df from the current RESULTS dict here rather
than trusting the Section-11 copy to still be complete.
```

**Original cell 101:**

```text
Bug fix: `best_row` comes from `results_df.loc[best_row_key]`, i.e. a single
*row* sliced across a DataFrame whose *columns* have mixed dtypes (floats like
f1/roc_auc, ints like n_params, lists like confusion matrices, etc.). Pandas has
no way to give that cross-column slice a single numeric dtype, so it comes back
as dtype=object even though every value we actually kept in cols_to_show is a
plain number -- and Series.round() refuses to run on an object-dtype Series
("Expected numeric dtype, got object instead"), even when every individual value
is numeric. Coerce that sub-selection to numeric first so .round() has something
it can work with; errors="coerce" turns any genuinely non-numeric entry into NaN
instead of crashing, which is the right behaviour for a display-only table.
```

**Original cell 115:**

```text
Bug fix: several thresholds commonly TIE for the minimum expected cost -- most
obviously when the validation set is separated cleanly (many QUICK_RUN / small-data
runs), where a whole wide band of thresholds all give zero cost. np.argmin() breaks
that tie by returning the FIRST (lowest) tied threshold, which just happens to sit at
one ragged edge of the safe band -- e.g. 0.17 -- rather than anywhere near its middle.
That edge threshold is far more likely to misclassify held-out test data than a
threshold from the middle of the same cost-minimising band, purely because it has much
less margin on one side. We instead take the MEDIAN threshold among every threshold
tied for the minimum cost, which sits centrally in the band and is the more defensible
'optimal' choice to report and deploy.
```

**Original cell 128:**

```text
Bug fix: OneVsRestClassifier.predict() thresholds each tactic's probability at a flat
0.5, exactly like Section 15.3's SOC classifier did before its cost-aware threshold
search. With short, sparse, templated text (very few non-stopword tokens survive per
email once the shared TF-IDF vocabulary is fit) and several genuinely rare tactics,
each per-tactic LogisticRegression's predict_proba rarely climbs above ~0.4-0.5 even
for true positives -- class_weight="balanced" reweights the *training loss*, it does
not guarantee predict_proba crosses 0.5 on unseen data. The fixed 0.5 cut used to
silently zero out every single prediction for every tactic (0.0 F1 across the board,
looking like the classifier had learned nothing, when predict_proba shows it clearly
had -- true positives were consistently scored far above true negatives, just below
0.5). Fixed the same way Section 15.3 fixes it for the main classifier: tune each
tactic's decision threshold on VALIDATION data (never test) to maximise validation F1,
then apply that per-tactic threshold to both val and test.
```

**Original cell 133:**

```text
Bug fix: this used to only fire when tld NOT IN ("com", "org") -- but .com is both the
single most common TLD used by real phishing domains AND the only TLD our own
synthetic examples use, so the two conditions were mutually exclusive by construction
and this heuristic could never fire on the very domains it was meant to catch (e.g.
"secure-1001-verify.example.com" tripped 0% of the time -- see Section 18.2's output
before this fix). A credential-themed keyword in the domain is suspicious on ANY TLD;
we now flag it unconditionally, and treat an uncommon TLD as an extra (stacking) reason.
```

**Original cell 156:**

```text
Bug fix: EnsembleStack has no model of its own to score raw text with --
it is a meta-classifier over OTHER finalists' probabilities (Section 12.2).
Previously this fell through to the generic NotImplementedError below, so
the adversarial robustness stress test (Section 21) silently skipped the
actual winning model and tested nothing. Fix: recursively score the same
perturbed text with each of the stack's own base models (whichever of
them score_texts can already handle -- classical/BiLSTM/hybrid all work;
a Transformer base model would raise here too, exactly as it does above),
then feed those probabilities through the already-fitted ENSEMBLE_META,
exactly mirroring how the stack is built and scored everywhere else in
the notebook (Section 12.2). This makes the stress test cover the model
that Section 14 actually selected as BEST_MODEL_NAME, not a fallback.
```

**Original cell 161:**

```text
Bug fix: tactic_clf.predict() uses a flat 0.5 cut per tactic, which Section 17.2
showed under-fires badly on this short/sparse text. Reuse the same per-tactic
validation-tuned thresholds (TACTIC_THRESHOLDS, Section 17.2) here instead.
```

**Original cell 192:**

```text
Bug fix: an ipywidgets `Output()` widget relies on the front-end picking up an
implicit display-update signal, which several notebook front-ends (Kaggle,
Colab, some JupyterLab/VS Code builds) don't reliably propagate for widgets
created and clicked within the same cell -- the click reaches the kernel
(the button itself works) but the report silently never paints. A plain
`HTML` widget instead renders on every `.value =` assignment, which every
front-end that can display ipywidgets at all is guaranteed to honour, so use
that as the primary render target. A small `Output()` is kept alongside it,
purely to catch and show a full traceback if something *else* goes wrong,
instead of failing silently a second way.
```

</details>

---

[← RAG, Latency, Deployment & LangGraph SOC Agent](06_rag_latency_deployment_soc_agent.ipynb) · **End of the study.** [Back to the start →](01_data_eda_and_leakage_safe_split.ipynb)